In [1]:
# Cell A — Imports, environment, and DB initialisation
import os
import sys
from pathlib import Path

from azure.identity import AzureCliCredential
from agent_framework.foundry import FoundryChatClient
from dotenv import load_dotenv

# Load .env from the day2-usecase root (one level up from notebooks/)
load_dotenv(Path("../").resolve() / ".env")

# Make the lib/ package importable
LIB_PATH = str(Path("../lib").resolve())
if LIB_PATH not in sys.path:
    sys.path.insert(0, LIB_PATH)

from ticket_management.database import init_db
from ticket_management.tools import (
    close_ticket,
    get_tickets_by_registered_by,
    get_tickets_by_resolved_by,
    register_ticket,
    resolve_ticket,
    search_tickets,
)

# Initialise the database (creates tables if not present)
init_db()
print("Database initialised.")
print("Tip: run  python -m lib.ticket_management.seed  from day2-usecase/ to load demo data.")

Database initialised.
Tip: run  python -m lib.ticket_management.seed  from day2-usecase/ to load demo data.


In [2]:
# Cell B — Azure AI Foundry client + ITSupportAgent
credential = AzureCliCredential()

client = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME"],
    credential=credential,
)

agent = client.as_agent(
    name="ITSupportAgent",
    instructions=(
        "You are a helpful IT support management agent for a company called DataTech.\n"
        "You assist both end users who want to raise IT support tickets and IT admin staff who resolve or close them.\n"
        "\n"
        "Guidelines:\n"
        "- Always use the provided tools — never fabricate ticket IDs or statuses.\n"
        "- When registering a ticket, confirm the ticket ID returned by the tool.\n"
        "- When resolving a ticket, ensure you capture who is resolving it and the resolution details.\n"
        "- When closing a ticket, confirm the final status from the tool response.\n"
        "- Address the user by name whenever it is known from the conversation.\n"
        "- Valid priorities: Low, Medium, High, Critical.\n"
        "- Valid statuses: Open, In Progress, On Hold, Resolved, Closed, Cancelled."
    ),
    tools=[
        register_ticket,
        get_tickets_by_registered_by,
        get_tickets_by_resolved_by,
        search_tickets,
        resolve_ticket,
        close_ticket,
    ],
)

print("ITSupportAgent ready.")

ITSupportAgent ready.


In [3]:
# Cell C — Create a shared session for all multi-turn turns
session = agent.create_session()
print("Session created. All scenario turns will share this session.")

Session created. All scenario turns will share this session.


In [4]:
# ---------------------------------------------------------------------------
# Scenario 1 — Jessie raises an IT support ticket
# ---------------------------------------------------------------------------
print("=" * 60)
print("SCENARIO 1: Jessie raises a ticket")
print("=" * 60)

result = await agent.run(
    "Hi, I'm Jessie Thompson. My laptop has completely stopped connecting to the office VPN since this morning. "
    "I've tried restarting multiple times but nothing works. This is blocking me from accessing all internal systems. "
    "Can you please raise a High priority support ticket for me?",
    session=session,
)
print(f"Agent: {result}\n")

SCENARIO 1: Jessie raises a ticket
Agent: Jessie, your support ticket has been successfully raised. Here are the details:

- **Ticket ID:** DTKT10016  
- **Description:** Laptop has stopped connecting to the office VPN since this morning. Tried restarting multiple times but issue persists, blocking access to internal systems.  
- **Priority:** High  
- **Status:** Open  

An IT technician will attend to this issue shortly. Let me know if you need any further assistance!



In [5]:
# ---------------------------------------------------------------------------
# Scenario 1 (follow-up) — Jessie asks for confirmation
# ---------------------------------------------------------------------------
result = await agent.run(
    "Thanks! Can you confirm the ticket ID and the current status for me?",
    session=session,
)
print(f"Agent: {result}\n")

Agent: Of course, Jessie! Here are the details again:

- **Ticket ID:** DTKT10016  
- **Current Status:** Open  

Let me know if you need further updates or assistance with anything else!



In [6]:
# ---------------------------------------------------------------------------
# Scenario 2 — IT Admin team resolves Jessie's ticket
# ---------------------------------------------------------------------------
print("=" * 60)
print("SCENARIO 2: IT Admin resolves the ticket")
print("=" * 60)

result = await agent.run(
    "This is the IT Admin team. I'm Aarav Sharma. "
    "We've investigated Jessie Thompson's VPN connectivity issue. "
    "Root cause: the VPN client certificate had expired and was not auto-renewed due to a misconfigured Group Policy. "
    "We've reissued the certificate, pushed the updated GPO, and verified that Jessie's laptop can now connect to the VPN successfully. "
    "Please mark Jessie's open VPN ticket as resolved with these details.",
    session=session,
)
print(f"Agent: {result}\n")

SCENARIO 2: IT Admin resolves the ticket
Agent: The ticket has been successfully marked as resolved, Aarav. Here are the updated details:

- **Ticket ID:** DTKT10016  
- **Description:** Laptop has stopped connecting to the office VPN since this morning.  
- **Priority:** High  
- **Status:** Resolved  
- **Resolved By:** Aarav Sharma  
- **Resolution Remarks:** The VPN client certificate had expired and was not auto-renewed due to a misconfigured Group Policy. Reissued the certificate, pushed the updated GPO, and verified successful VPN connectivity on Jessie's laptop.  

Let me know if there's anything else you'd like me to do!



In [7]:
# ---------------------------------------------------------------------------
# Scenario 3 — IT Admin closes the resolved ticket
# ---------------------------------------------------------------------------
print("=" * 60)
print("SCENARIO 3: IT Admin closes the ticket")
print("=" * 60)

result = await agent.run(
    "IT Admin here again — Aarav Sharma. "
    "Jessie has confirmed that the VPN is working perfectly now. "
    "Please go ahead and close that resolved VPN ticket.",
    session=session,
)
print(f"Agent: {result}\n")

SCENARIO 3: IT Admin closes the ticket
Agent: The ticket has been successfully closed, Aarav. Here are the final details:

- **Ticket ID:** DTKT10016  
- **Description:** Laptop has stopped connecting to the office VPN since this morning.  
- **Priority:** High  
- **Status:** Closed  
- **Resolved By:** Aarav Sharma  
- **Resolution Remarks:** The VPN client certificate had expired and was not auto-renewed due to a misconfigured Group Policy. Reissued the certificate, pushed the updated GPO, and verified successful VPN connectivity on Jessie's laptop.  

Let me know if there’s anything else I can assist with!



In [8]:
# ---------------------------------------------------------------------------
# Scenario 3 — IT Admin closes the resolved ticket
# ---------------------------------------------------------------------------
print("=" * 60)
print("SCENARIO 4: IT Admin Queries all Open ticket")
print("=" * 60)

result = await agent.run(
    "Before we wrap up, can you provide me with a list of all currently open tickets in the system?",
    session=session,
)
print(f"Agent: {result}\n")

SCENARIO 4: IT Admin Queries all Open ticket
Agent: Here’s the list of all currently open tickets in the system:

1. **Ticket ID:** DTKT10015  
   **Description:** Developer laptop runs extremely slow after the latest macOS Sequoia update.  
   **Registered By:** Ethan Davis  
   **Priority:** High  
   **Status:** Open  

2. **Ticket ID:** DTKT10012  
   **Description:** USB peripherals (keyboard and mouse) not recognised after docking station firmware update.  
   **Registered By:** Ravi Menon  
   **Priority:** Medium  
   **Status:** Open  

3. **Ticket ID:** DTKT10009  
   **Description:** Corporate email signature template not rendering correctly in Outlook Web.  
   **Registered By:** Jessie Thompson  
   **Priority:** Low  
   **Status:** Open  

4. **Ticket ID:** DTKT10006  
   **Description:** Two-factor authentication app not syncing with the company SSO portal.  
   **Registered By:** Charlotte Brown  
   **Priority:** High  
   **Status:** Open  

5. **Ticket ID:** DTKT100

In [9]:
# ---------------------------------------------------------------------------
# Cell H — Verification: read Jessie's ticket directly from the DB
# ---------------------------------------------------------------------------
import json
from ticket_management.repository import get_tickets_by_registered_by

print("=" * 60)
print("VERIFICATION: Final state of Jessie Thompson's tickets")
print("=" * 60)

jessie_tickets = get_tickets_by_registered_by("Jessie Thompson")
for t in jessie_tickets:
    print(json.dumps(t, indent=2))
    print()

VERIFICATION: Final state of Jessie Thompson's tickets
{
  "ticket_id": "DTKT10016",
  "ticket_description": "Laptop has stopped connecting to the office VPN since this morning. Tried restarting multiple times but issue persists, blocking access to internal systems.",
  "registered_date": "2026-05-05T22:29:59.717035",
  "registered_by": "Jessie Thompson",
  "priority": "High",
  "status": "Closed",
  "resolved_by": "Aarav Sharma",
  "resolution_remarks": "The VPN client certificate had expired and was not auto-renewed due to a misconfigured Group Policy. Reissued the certificate, pushed the updated GPO, and verified successful VPN connectivity on Jessie's laptop.",
  "resolved_date": "2026-05-05T22:30:36.023392"
}

{
  "ticket_id": "DTKT10009",
  "ticket_description": "Corporate email signature template not rendering correctly in Outlook Web.",
  "registered_date": "2026-05-02T22:24:41.871562",
  "registered_by": "Jessie Thompson",
  "priority": "Low",
  "status": "Open",
  "resolved_b